### How to Avoid Path Errors When Moving Your External Hard Disk

Because you are using an external hard disk, Windows will frequently change its drive letter (switching between D:\, E:\, F:\, etc.), which breaks hardcoded paths.

To permanently stop this from happening, use a dynamic project_config.py file placed inside your notebooks/ directory.

### Python Script to Rename Raw Images & Update manifest.csv

In [2]:
import os
import sys
from pathlib import Path
import pandas as pd

# Import dynamically resolved paths from project_config.py
try:
    from project_config import MANIFEST_PATH, RAW_ROOT, PROJECT_ROOT
except ImportError:
    sys.path.append(str(Path.cwd()))
    from project_config import MANIFEST_PATH, RAW_ROOT, PROJECT_ROOT

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"manifest.csv not found at dynamic path: {MANIFEST_PATH}")

print(f"Loading manifest from: {MANIFEST_PATH}")
df = pd.read_csv(MANIFEST_PATH)

counters = {}
new_filepaths = []
renamed_count = 0

print("Starting dataset image renaming and manifest update...")

for idx, row in df.iterrows():
    raw_path_str = str(row["filepath"])
    
    # 1. Dynamically fix old drive letters (e.g., G:\ or E:\) to current project environment
    # Extract everything after "Final Datasets" to keep the relative subfolders intact
    if "Final Datasets" in raw_path_str:
        relative_part = raw_path_str.split("Final Datasets")[-1].lstrip("\\/")
        old_path = RAW_ROOT / relative_part
    else:
        old_path = Path(raw_path_str)
    
    if not old_path.exists():
        # Fallback check if path is already correct as-is
        old_path_direct = Path(raw_path_str)
        if old_path_direct.exists():
            old_path = old_path_direct
        else:
            # Keep path as-is if the file cannot be found physically anywhere
            new_filepaths.append(raw_path_str)
            continue

    # 2. Extract dataset folder name from path components
    parts = old_path.parts
    dataset_folder_name = parts[-3] if len(parts) >= 3 else "Dataset"
    
    # Generate initials: e.g., "Cotton Leaf Closed Environment" -> "CLCE"
    initials = "".join([word[0].upper() for word in dataset_folder_name.split() if word])
    
    # Get class name and format it safely
    class_name = row.get("disease_raw", parts[-2])
    safe_class_name = str(class_name).replace(" ", "_")
    
    # Maintain sequential count per class per dataset
    key = (initials, safe_class_name)
    if key not in counters:
        counters[key] = 1
    else:
        counters[key] += 1
    
    count_num = counters[key]
    
    # Format: <Initials>_<ClassName>_<Count>.jpg (e.g., CLCE_Alternaria_Leaf_Spot_1.png)
    new_filename = f"{initials}_{safe_class_name}_{count_num}{old_path.suffix}"
    new_path = old_path.parent / new_filename
    
    # Rename physical file on disk
    if old_path != new_path:
        if new_path.exists():
            while new_path.exists():
                count_num += 1
                new_filename = f"{initials}_{safe_class_name}_{count_num}{old_path.suffix}"
                new_path = old_path.parent / new_filename
        
        old_path.rename(new_path)
        renamed_count += 1
        
    new_filepaths.append(str(new_path))

# Update dataframe and save back to the dynamic manifest path
df["filepath"] = new_filepaths
df.to_csv(MANIFEST_PATH, index=False)

print(f"Done! Successfully renamed {renamed_count} files on disk and updated {MANIFEST_PATH}.")

Loading manifest from: F:\Crop Identification\notebooks\manifest.csv
Starting dataset image renaming and manifest update...
Done! Successfully renamed 30414 files on disk and updated F:\Crop Identification\notebooks\manifest.csv.
